### Import the libraries

In [3]:
from binance.client import Client
import pandas as pd
import numpy as np

### Instantiate the Binance client and download the last 500 BTCEUR trades

In [11]:
spot_client = Client()
trades = spot_client.get_recent_trades(symbol="BTCUSDT")

df = pd.DataFrame(trades)


### Process the downloaded trades into a pandas DataFrame

In [12]:
df.drop(columns=["id", "isBuyerMaker", "isBestMatch"], inplace=True)
df["time"] = pd.to_datetime(df["time"], unit="ms")
for column in ["price", "qty", "quoteQty"]:
    df[column] = pd.to_numeric(df[column])

In [13]:
df.head()

,price,qty,quoteQty,time
0,75191.04,0.00095,71.431488,2026-04-19 07:47:01.035
1,75191.04,0.19589,14729.172826,2026-04-19 07:47:01.978
2,75191.04,0.19936,14990.085734,2026-04-19 07:47:01.978
3,75191.04,0.04033,3032.454643,2026-04-19 07:47:01.978
4,75191.04,0.19385,14575.783104,2026-04-19 07:47:01.978


###  Define a function aggregating the raw trades information into bars

In [16]:
def get_bars(df, add_time=False):
    ohlc = df["price"].ohlc()
    vwap = (
        df.apply(lambda x: np.average(x["price"], weights=x["qty"]))
        .to_frame("vwap")
    )
    vol = df["qty"].sum().to_frame("vol")
    cnt = df["qty"].size().to_frame("cnt")
    if add_time:
        time = df["time"].last().to_frame("time")
        res = pd.concat([time, ohlc, vwap, vol, cnt], axis=1)
    else:
        res = pd.concat([ohlc, vwap, vol, cnt], axis=1)
    return res

### Get the time bars

In [17]:
df_grouped_time = df.groupby(pd.Grouper(key="time", freq="1Min"))
time_bars = get_bars(df_grouped_time)
time_bars

C:\Users\MSI VN\AppData\Local\Temp\ipykernel_10032\2030430829.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.apply(lambda x: np.average(x["price"], weights=x["qty"]))


,open,high,low,close,vwap,vol,cnt
time,,,,,,,
2026-04-19 07:47:00,75191.04,75191.04,75177.63,75178.62,75186.38713,1.59093,500


### Get the tick bars

In [18]:
bar_size = 50
df["tick_group"] = (
    pd.Series(list(range(len(df))))
    .div(bar_size)
    .apply(np.floor)
    .astype(int)
    .values
)
df_grouped_ticks = df.groupby("tick_group")
tick_bars = get_bars(df_grouped_ticks, add_time=True)
tick_bars


C:\Users\MSI VN\AppData\Local\Temp\ipykernel_10032\2030430829.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.apply(lambda x: np.average(x["price"], weights=x["qty"]))


,time,open,high,low,close,vwap,vol,cnt
tick_group,,,,,,,,
0,2026-04-19 07:47:04.146,75191.04,75191.04,75191.02,75191.02,75191.038032,0.86943,50
1,2026-04-19 07:47:04.666,75191.02,75191.02,75186.05,75186.94,75189.263855,0.04296,50
2,2026-04-19 07:47:05.438,75186.94,75186.94,75186.00,75186.00,75186.903550,0.09967,50
3,2026-04-19 07:47:05.446,75185.61,75185.61,75183.71,75183.71,75184.413103,0.03387,50
4,2026-04-19 07:47:05.456,75183.68,75183.68,75183.56,75183.56,75183.571111,0.00351,50
5,2026-04-19 07:47:05.458,75183.56,75183.56,75179.50,75179.50,75181.123043,0.01403,50
6,2026-04-19 07:47:07.751,75179.30,75179.30,75179.07,75179.07,75179.086897,0.06891,50
7,2026-04-19 07:47:07.771,75179.07,75179.07,75178.00,75178.00,75178.621516,0.00686,50
8,2026-04-19 07:47:10.221,75178.61,75178.62,75177.63,75178.62,75178.608089,0.11376,50


### Get the volume bars

In [19]:
bar_size = 1
df["cum_qty"] = df["qty"].cumsum()
df["vol_group"] = (
    df["cum_qty"]
    .div(bar_size)
    .apply(np.floor)
    .astype(int)
    .values
)
df_grouped_ticks = df.groupby("vol_group")
volume_bars = get_bars(df_grouped_ticks, add_time=True)
volume_bars

C:\Users\MSI VN\AppData\Local\Temp\ipykernel_10032\2030430829.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.apply(lambda x: np.average(x["price"], weights=x["qty"]))


,time,open,high,low,close,vwap,vol,cnt
vol_group,,,,,,,,
0,2026-04-19 07:47:05.435,75191.04,75191.04,75186.05,75186.94,75190.699247,0.97434,118
1,2026-04-19 07:47:20.258,75186.94,75186.94,75177.63,75178.62,75179.573092,0.61659,382


### Get the dollar bars

In [20]:
bar_size = 50000
df["cum_value"] = df["quoteQty"].cumsum()
df["value_group"] = (
    df["cum_value"]
    .div(bar_size)
    .apply(np.floor)
    .astype(int)
    .values
)
df_grouped_ticks = df.groupby("value_group")
dollar_bars = get_bars(df_grouped_ticks, add_time=True)
dollar_bars

C:\Users\MSI VN\AppData\Local\Temp\ipykernel_10032\2030430829.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.apply(lambda x: np.average(x["price"], weights=x["qty"]))


,time,open,high,low,close,vwap,vol,cnt
value_group,,,,,,,,
0,2026-04-19 07:47:01.978,75191.04,75191.04,75191.04,75191.04,75191.040000,0.63038,5
1,2026-04-19 07:47:10.706,75191.04,75191.04,75177.63,75178.61,75185.803888,0.63053,447
2,2026-04-19 07:47:20.258,75178.61,75178.62,75178.61,75178.62,75178.613890,0.33002,48
